In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
import sys
import os

# Add the directory containing the file to the system path
sys.path.append(os.path.abspath("../module1"))

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

Start with one ground truth record:

In [11]:
q = ground_truth[0]
q

{'question': 'I just found this course, is it too late for me to join, or can I still sign up?',
 'document': '74eb249bbf'}

In [12]:
doc_id = q["document"]
results = text_search(query=q["question"]) 

In [13]:
doc_id

'74eb249bbf'

In [14]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '85384a18e5',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'OpenAI: Do I have to subscribe and pay for Open AI API for this course?',
  'answer': "No, you don't have to pay for this service in o

First, compare the retrieved document IDs with the correct document ID:

In [15]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [16]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [17]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [19]:
q = ground_truth[0]
print(q["question"])          
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course, is it too late for me to join, or can I still sign up?


[1, 0, 0, 0, 0]

In [30]:
q = ground_truth[19]
print(q["question"])
compute_relevance_text(q)
# print()
# [1, 0, 0, 0, 0]

Is the homework basically the same as the lesson exercises, or does it use a different dataset/task?


[0, 0, 0, 0, 0]

In [32]:
q = ground_truth[51]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

What’s the best place to follow homework deadlines and my progress in LLM Zoomcamp?


[1, 0, 0, 0, 0]

In [33]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [34]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [35]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [36]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [37]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [44]:
relevance_15 = compute_relevance_total(ground_truth_sample, text_search)
relevance_15

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [45]:
len(relevance_15)

15

In [40]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

In [41]:
len(relevance_total)

565

Hit Rate

In [48]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [50]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [51]:
cnt / len(example)
# 0.933

0.9333333333333333

In [52]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [53]:
hit_rate(example)
# 0.933

0.9333333333333333

Mean Reciprocal Rank (MRR)

MRR also considers the position.

For each query, the score is based on the rank of the first correct document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0

In the example, most hits are at the first position. Some hits are lower in the list.

Look at that line:

In [54]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

For above line, the score is 1/2 because the correct document is at position 2

In [56]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [57]:
total_score / len(example)
# 0.822

0.8222222222222222

MRR is the average of these scores across all queries. It rewards systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually smaller because some correct documents are found below the first position.

Put the same logic into a function:



In [58]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [59]:
mrr(example)
# 0.822

0.8222222222222222

In [60]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [61]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8353982300884956, 'mrr': 0.7246607669616517}

Interpreting the metrics
A few things to keep in mind when reading these numbers:

Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant. A 50% hit rate does not mean that half the results are useless. It means the document we generated the question from did not appear in the top results for half the queries. Other relevant documents may still be there.

With synthetic data, the generated questions can be too close to the original FAQ text. This inflates hit rate and MRR. If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher. The right number depends on how much the downstream LLM can compensate for imperfect retrieval. It also depends on user tolerance for wrong answers.

Look at the system holistically. A high MRR means the relevant document is near the top, which helps the LLM focus on the right context. A low MRR with a high hit rate means the document is there, but buried under less relevant results.



**Trying different boosts**

In [ ]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )   

In [63]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/565 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8831858407079646, 'mrr': 0.7955752212389379}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8761061946902655, 'mrr': 0.7756342182890853}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8353982300884956, 'mrr': 0.7246607669616517}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.815929203539823, 'mrr': 0.6905899705014746}


  0%|          | 0/565 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8, 'mrr': 0.662300884955752}


In [64]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [65]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

In [66]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
35,5.0,10.0,0.5,0.968142,0.873097
3,1.0,2.0,0.1,0.968142,0.873097
19,2.0,4.0,0.2,0.968142,0.873097
18,2.0,4.0,0.1,0.962832,0.871327
34,5.0,10.0,0.2,0.962832,0.870295
33,5.0,10.0,0.1,0.962832,0.870000
7,1.0,4.0,0.2,0.968142,0.870000
6,1.0,4.0,0.1,0.968142,0.869794
4,1.0,2.0,0.2,0.964602,0.867611
8,1.0,4.0,0.5,0.964602,0.867316


The best combination weights answer twice as heavily as question, with almost no weight on section. So the data says the opposite of where we started. The answer text matters more for retrieval than the question text. The intuition was wrong, and we'd never have known without measuring it. This is exactly why we evaluate instead of guess.

The first three rows have the same relative weights:

question : answer : section = 1 : 2 : 0.1

So we can use the smaller and easier-to-read values: question=1.0, answer=2.0, and section=0.1. This gives the same relative weights as question=5.0, answer=10.0, and section=0.5, but the numbers are not unnecessarily large.

Define the search function with these boosts:

In [68]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Usually we care about both metrics. Hit Rate tells us whether the correct document appears at all. MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.



Tuning Workflow
Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt. It explores the parameter space more efficiently by focusing on combinations that are likely to improve the metric.

Top-K tradeoffs
We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. Five results is a reasonable default for short FAQ-style documents.

Next, we'll move from retrieval quality to answer quality and evaluate the full RAG pipeline.